In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_rows', 25)
%matplotlib inline

from src.data_loading import load_raw_tables, build_transactions
from src.cleaning import clean_transactions
from src.categorization import categorize
from src.config import DATA_DIR
from src.eda import (
    top_products, plot_top_products,
    pareto_analysis, plot_pareto,
    revenue_by_category, plot_category_breakdown,
    revenue_by_month, plot_monthly_revenue,
    revenue_by_weekday, plot_weekday,
    revenue_by_hour, plot_hourly,
    category_seasonality, plot_category_seasonality,
    revenue_by_store, plot_store_revenue,
    store_category_mix, plot_store_category_mix,
)
from src.data_quality import (
    unknown_product_summary, unknown_by_store, unknown_over_time,
    unknown_store_over_time, plot_unknown_by_store,
    plot_unknown_over_time, plot_unknown_store_over_time,
)

tables = load_raw_tables(DATA_DIR)
df = categorize(clean_transactions(build_transactions(tables)))
print(f'Ready: {len(df):,} rows  |  {df["name"].nunique()} unique products  |  {df["name_category"].nunique()} categories')

## Block A — Sales EDA
### 1. Top Products by Revenue

In [ ]:
tbl_rev = top_products(df, by='revenue', n=20)
tbl_rev

In [ ]:
fig = plot_top_products(df, by='revenue', n=15)
fig

### 2. Top Products by Units Sold

In [ ]:
tbl_units = top_products(df, by='units', n=15)
tbl_units

In [ ]:
fig = plot_top_products(df, by='units', n=15)
fig

### 3. Pareto Analysis (80 / 20 Rule)

In [ ]:
prod, n80 = pareto_analysis(df)
total_rev = df['revenue'].sum()
top20_share = prod.head(20)['revenue'].sum() / total_rev * 100
print(f'Products to reach 80% of revenue: {n80}')
print(f'Top-20 products share:            {top20_share:.1f}%  (anchor: ~80%)')
prod.head(25)

In [ ]:
fig, n80 = plot_pareto(df)
fig

### 4. Revenue by Category

In [ ]:
cat_tbl = revenue_by_category(df)
cat_tbl

In [ ]:
fig = plot_category_breakdown(df)
fig

### 5. Monthly Revenue (Seasonality)

In [ ]:
month_tbl = revenue_by_month(df)
peak = month_tbl.loc[month_tbl['revenue'].idxmax()]
print(f'Peak month: {peak["month_label"]}  ({peak["order_count"]:,.0f}% of orders, revenue index {peak["revenue"]:.0f})')
month_tbl

In [ ]:
fig = plot_monthly_revenue(df)
fig

### 6. Revenue by Day of Week (local time)

In [ ]:
wd_tbl = revenue_by_weekday(df)
print('Busiest days (revenue index, Mon=0):')
for _, r in wd_tbl.sort_values('revenue', ascending=False).iterrows():
    print(f'  {r["weekday_name"]:<12}  idx={r["revenue"]:>5.1f}  orders={r["order_count"]:>5.1f}%')
wd_tbl

In [ ]:
fig = plot_weekday(df)
fig

### 7. Revenue by Hour of Day (local time)

In [ ]:
hr_tbl = revenue_by_hour(df)
top3_hours = hr_tbl.nlargest(3, 'revenue')
print('Top 3 revenue hours (local, revenue index):')
for _, r in top3_hours.iterrows():
    print(f'  {r["hour"]:02d}:00   idx={r["revenue"]:>5.1f}  orders={r["order_count"]:>5.2f}%')
hr_tbl

In [ ]:
fig = plot_hourly(df)
fig

### 8. Category Seasonality Heatmap

In [ ]:
fig = plot_category_seasonality(df)
fig

### 9. Store Performance

In [ ]:
store_tbl = revenue_by_store(df)
store_tbl[['store_label','revenue','order_count','avg_order_value']]

In [ ]:
fig = plot_store_revenue(df)
fig

### 10. Store × Category Mix

In [ ]:
mix = store_category_mix(df)
mix.round(1)

In [ ]:
fig = plot_store_category_mix(df)
fig

### Validation Anchors

In [ ]:
top1 = tbl_rev.iloc[0]
assert top1['name'] == '2 kugler'
print(f'Top product:  {top1["name"]}  —  {top1["revenue"]:.1f}% of revenue  [anchor: ~20%]')

peak_month = month_tbl.loc[month_tbl['revenue'].idxmax(), 'month_label']
assert any(m in peak_month for m in ('May', 'Jun', 'Aug')), f'Unexpected peak: {peak_month}'
print(f'Peak month:   {peak_month}  [anchor: May-Jun]')

top2_days = wd_tbl.nlargest(2, 'revenue')['weekday_name'].tolist()
assert any(d in top2_days for d in ('Friday', 'Saturday'))
print(f'Top-2 days:   {top2_days}  [anchor: Fri/Sat]')

top3_hrs = hr_tbl.nlargest(3, 'revenue')['hour'].tolist()
assert any(h in top3_hrs for h in range(11, 16))
print(f'Top-3 hours:  {[f"{h}:00" for h in sorted(top3_hrs)]}  [anchor: 12-15]')

print('\nAll validation anchors passed.')

---
## Data Quality — 'Unknown Product'

Rows where `title == 'Unknown Product'` indicate a failed join between `line_items.product_id`
and the Shopify products catalog. The `name` field (POS label) is always present and is used
for all categorization, so this issue does not affect revenue or category analysis — but it
does signal a POS configuration gap worth documenting.

In [ ]:
summary = unknown_product_summary(df)
print('=== Overall unknown-product rate ===')
print(summary.to_string(index=False))
print()
print('Note: the unit rate is ~12x the row rate because unknown-product lines')
print('tend to be bulk/box transactions with high per-line quantities.')

In [ ]:
store_dq = unknown_by_store(df)
print('=== Unknown-product rate by store ===')
print(store_dq[['store_label','unknown_rows','total_rows','row_rate',
                'unknown_units','total_units','unit_rate']].to_string(index=False))
worst = store_dq.iloc[0]
print(f'\nMost-affected store: {worst["store_label"]}')
print(f'  Unit rate: {worst["unit_rate"]*100:.1f}%  vs next: {store_dq.iloc[1]["unit_rate"]*100:.1f}%')

In [ ]:
fig = plot_unknown_by_store(df)
fig

In [ ]:
monthly_dq = unknown_over_time(df)
print('=== Monthly unknown-product rate (all stores) ===')
print(monthly_dq[['month_label','total_rows','unknown_rows','row_rate',
                   'total_units','unknown_units','unit_rate']].to_string(index=False))

In [ ]:
fig = plot_unknown_over_time(df)
fig

In [ ]:
store_monthly, worst_label = unknown_store_over_time(df)
print(f'=== Monthly unknown-product rate — {worst_label} (most affected) ===')
print(store_monthly[['month_label','total_rows','unknown_rows','row_rate',
                      'total_units','unknown_units','unit_rate']].to_string(index=False))

In [ ]:
fig, worst_label = plot_unknown_store_over_time(df)
fig

In [ ]:
# --- Findings summary ---
row_rate  = summary.loc[0, 'rate_pct']
unit_rate = summary.loc[1, 'rate_pct']
drop_month = store_monthly.loc[store_monthly['unit_rate'] < 0.01, 'month_label']
drop_from  = drop_month.iloc[0] if len(drop_month) else 'not yet resolved'

print('=== Findings ===')
print(f'Overall rate:          {row_rate:.2f}% of rows, {unit_rate:.2f}% of units')
print(f'Concentrated in:       {worst_label} ({worst["unit_rate"]*100:.1f}% unit rate)')
print(f'Near-zero from:        {drop_from} in {worst_label} (row rate & unit rate both <1%)')
print()
print('Interpretation: a POS terminal was entering bulk transactions without a')
print('Shopify-linked SKU, inflating the unit rate. The issue largely resolved')
print(f'after {drop_from}, consistent with a POS reconfiguration or catalog update.')